[关于pytorch中使用detach并不能阻止参数更新这档子事儿](https://zhuanlan.zhihu.com/p/344916574)

一般网络，只关一次，没问题

In [15]:
import torch
import numpy as np
import torch.nn as nn
class TestDetach(nn.Module):
    def __init__(self, InDim, HiddenDim, OutDim):
        super().__init__()
        self.layer1 = nn.Linear(InDim, HiddenDim, False)
        self.layer2 = nn.Linear(HiddenDim, OutDim, False)

    def forward(self, x, DetachLayer1):
        x = torch.relu(self.layer1(x))
        x = x.detach()
        print(x.grad)
        x = self.layer2(x)
        return x

In [25]:
def train():
    # 随机生成数据
    N, F = 5000, 196
    x = torch.Tensor(np.random.randn(N, F))
    y = torch.LongTensor(np.random.randint(0, 3, (N,)))
    # 生成损失函数，模型，优化器
    LossFunc = nn.CrossEntropyLoss()
    model = TestDetach(F, 64, 3)
    optimizer = torch.optim.Adam(model.parameters(), lr=1, weight_decay=0.5)
    # 训练
    for epoch in range(3):
        # 将DetachLayer1交替置为True和False
        DetachLayer1 = True if epoch % 2 == 0 else False
        Yhat = model(x, DetachLayer1)
        Loss = LossFunc(Yhat, y)
        print(f"Epoch {epoch}, DetachLayer1: {DetachLayer1}, Loss: {Loss:.4f}")
        Loss.backward()
        print(model.layer1.weight.data[0, :5])
        print(model.layer1.weight.grad)
        optimizer.step()
        print(model.layer1.weight.data[0, :5])
        optimizer.zero_grad()
        print()

In [26]:
train()

None
Epoch 0, DetachLayer1: True, Loss: 1.1343
tensor([ 0.0239, -0.0039, -0.0132, -0.0225,  0.0708])
None
tensor([ 0.0239, -0.0039, -0.0132, -0.0225,  0.0708])

Epoch 1, DetachLayer1: False, Loss: 4.5386
tensor([ 0.0239, -0.0039, -0.0132, -0.0225,  0.0708])
tensor([[-2.5846e-03,  6.6263e-02, -1.0738e-02,  ..., -3.5262e-02,
         -3.6462e-02,  5.1060e-02],
        [-1.1827e-05, -9.5051e-04,  1.2657e-03,  ...,  1.4711e-03,
         -8.5802e-04,  5.0927e-04],
        [-8.4335e-04,  5.9634e-02, -4.3369e-02,  ..., -6.1002e-02,
          9.0589e-03,  8.3038e-03],
        ...,
        [ 1.0488e-03, -6.3993e-02,  4.3269e-02,  ...,  6.2622e-02,
         -5.6594e-03, -1.2563e-02],
        [-1.6107e-03,  6.7774e-03,  2.9507e-02,  ...,  2.2970e-02,
         -4.1752e-02,  3.9409e-02],
        [ 9.2122e-04, -6.0218e-02,  4.2211e-02,  ...,  6.0226e-02,
         -7.1817e-03, -1.0153e-02]])
tensor([-0.9761, -1.0039,  0.9868, -1.0225, -0.9292])

None
Epoch 2, DetachLayer1: True, Loss: 110.6902
tensor

可以看到权重不变， `tensor([-0.0485, -0.0191, -0.0404, -0.0675, -0.0586]`。
权重的梯度一直是None

复杂网络，时开时关的detach，就不能组织权重更新

In [27]:
class TestDetach(nn.Module):
    def __init__(self, InDim, HiddenDim, OutDim):
        super().__init__()
        self.layer1 = nn.Linear(InDim, HiddenDim, False)
        self.layer2 = nn.Linear(HiddenDim, OutDim, False)

    def forward(self, x, DetachLayer1):  # 多传了一个bool参数，以指示是否detach
        x = torch.relu(self.layer1(x))
        if DetachLayer1:
            x = x.detach()
            print(x.grad)
        x = self.layer2(x)
        return x

In [28]:
train()

None
Epoch 0, DetachLayer1: True, Loss: 1.1169
tensor([ 0.0368, -0.0044,  0.0691, -0.0247, -0.0681])
None
tensor([ 0.0368, -0.0044,  0.0691, -0.0247, -0.0681])

Epoch 1, DetachLayer1: False, Loss: 3.7539
tensor([ 0.0368, -0.0044,  0.0691, -0.0247, -0.0681])
tensor([[-4.2477e-04,  1.6615e-02,  4.0042e-02,  ...,  1.1851e-02,
         -1.7905e-02, -1.9043e-02],
        [-1.2092e-05,  1.4711e-03,  1.8231e-04,  ...,  2.0018e-04,
         -5.2038e-04, -4.0706e-04],
        [ 5.0156e-02,  9.1190e-03, -9.5578e-03,  ..., -2.2554e-02,
         -2.1279e-02,  3.5934e-02],
        ...,
        [-1.9774e-02,  4.6852e-03,  4.6289e-02,  ...,  4.6738e-02,
          1.9660e-02, -2.0597e-02],
        [-1.8527e-02, -2.2335e-02, -3.8626e-02,  ..., -1.0806e-02,
          3.4251e-03,  7.3777e-03],
        [ 1.4350e-02, -4.5290e-03, -1.1643e-02,  ..., -4.1449e-02,
          1.7591e-02, -4.6811e-03]])
tensor([-0.9632, -1.0044, -0.9309,  0.9753,  0.9319])

None
Epoch 2, DetachLayer1: True, Loss: 35.7086
tensor(

在Epoch2中，网络中的`print(x.grad)`是None，但是训练中的权重的梯度就只是置0，而不是None，从而最后权重更新了。

![图 1](../images/49a09cc4ea18ebc865d62350ddfb5ff0f0325b74ceea1cef6c800f3fe3fcb035.png)
这个人纯纯在放屁。